# Daily Challenge: Pinecone Serverless Reranking in Action

Reranking models boost search relevance by assigning similarity scores between a query and documents, then reordering results so the most pertinent information appears first. In contexts like healthcare, this helps clinicians quickly access the most critical clinical notes.

> **Prerequisite:** A Pinecone account and an API key. Sign up at [pinecone.io](https://www.pinecone.io/) if you haven't already.

## Part 1: Load Documents & Execute Reranking Model

### 1. Install Pinecone libraries
If you get version conflicts, restart your runtime after installation.

In [1]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 26.3 MB/s eta 0:00:00


### 2. Authenticate with Pinecone
Securely provide your API key so the client can connect without hard-coding secrets. Get the key from the Pinecone dashboard under "API Keys".

In [4]:
import os

if not os.environ.get("pcsk_4e5TE7_AoUoBrAKs86ScQkk9RVNmTdkcns8c1CgRqJpUwGV3zxG2dr1THsBydDpNFp6tc4"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

### 3. Instantiate the Pinecone client
The client (`pc`) is your entry point for all Pinecone operations — creating indexes, querying, and reranking.

In [5]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

### 4. Define your query & documents
A small set of documents that mix references to **Apple** (the company) and **apple** (the fruit) to test the reranker's contextual understanding.

In [6]:
query = "Tell me about Apple's products"
documents = [
    "An apple is a sweet, edible fruit produced by the apple tree (Malus domestica), commonly eaten raw or baked into pies.",  # apple fruit
    "Apple Inc. designs and sells the iPhone, iPad, Mac, Apple Watch, and AirPods, along with services like iCloud and Apple Music.",  # Apple company products
    "Apples come in many varieties such as Gala, Fuji, and Granny Smith, and are a great source of dietary fiber and vitamin C.",  # another fruit document
    "Apple's M-series chips power the latest MacBook Pro and Mac Studio, delivering high performance with low energy consumption.",  # another company document
    "The Swiss canton of Geneva is famous for its lake, fountains, and watchmaking heritage."  # unrelated document (my choice)
]

### 5. Call the reranker
`top_n` limits the number of reranked results so you only retrieve the most relevant documents.

In [7]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,
)

### 6. Inspect reranked results
Higher scores mean more relevant. The reranked object exposes its ordered results on the `.data` attribute.

In [8]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i + 1}. score={m.score:.4f} | {m.document.text}")

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
1. score=0.7870 | Apple Inc. designs and sells the iPhone, iPad, Mac, Apple Watch, and AirPods, along with services like iCloud and Apple Music.
2. score=0.1272 | An apple is a sweet, edible fruit produced by the apple tree (Malus domestica), commonly eaten raw or baked into pies.
3. score=0.1000 | Apple's M-series chips power the latest MacBook Pro and Mac Studio, delivering high performance with low energy consumption.


## Part 2: Setup a Serverless Index for Medical Notes

### 1. Install data & model libraries

In [9]:
!pip install pandas torch transformers

### 2. Import modules & define environment settings
Most Pinecone accounts use `aws` and `us-east-1`.

In [10]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Cloud and region settings (defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'

### 3. Create or recreate the index
The index dimension must match the embedding model (`all-MiniLM-L6-v2` outputs **384**-dim vectors). Cosine similarity works well for text embeddings.

In [11]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec,
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-zv4821l.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

## Part 3: Load the Sample Data

### 1. Download & read JSONL
Downloads sample medical notes data that's already been processed and embedded. Use the **raw** GitHub URL.

In [13]:
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

### 2. Preview the DataFrame
`df.shape` shows the (rows, columns) dimensions.

In [14]:
# Show shape of the DataFrame
print("Data shape:", df.shape)
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


## Part 4: Upsert Data into the Index

### 1. Instantiate index client & upsert

In [15]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

### 2. Wait for availability
We wait until there is at least one vector indexed, so the count must be greater than `0`.

In [16]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

## Part 5: Query & Embedding Function

### 1. Define your embedding function
Average across the **sequence length** dimension (`dim=0` of `last_hidden_state[0]`, shape `[seq_len, hidden]`) to get a single vector per input.

In [17]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0)  # average over sequence length
    return embedding

### 2. Run a semantic search query
Retrieves the most semantically similar notes for a clinical query.

In [18]:
# Build a query to search
question = "patient has chest pain"
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=10, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Part 6: Display & Rerank Clinical Notes

### 1. Display initial search results
Pinecone query matches expose the similarity score under `score` and the metadata under `metadata`.

In [19]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

Question: 'patient has chest pain'

Results:
   1. ID: P001
 Score: 0.734460115
 Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
 Score: 0.483538181
 Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P0100
 Score: 0.446577698
 Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

   4. ID: P095
 Score: 0.416666359
 Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   5. ID: P047
 Score: 0.416666359
 Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   6. ID: P003
 Score: 0.412105769
 Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   7. ID: P063
 Score: 0.385879129
 Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   8. ID: P090
 Score: 0.37495032
 Metadata: {'advice': 'stress management', 'symptoms': 'stress, burnout'}

   9. ID: P042
 Score: 0.37495032


### 2. Prepare documents for reranking
Concatenate each note's metadata into a single `reranking_field` string for the reranker to rescore against.

In [20]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

### 3. Execute serverless reranking
Reranking uses a refined query and the metadata field to reorder notes by their new relevance scores.

In [21]:
# Define a more specific query for reranking
refined_query = "patient needs knee surgery"

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

### 4. Show reranked results
Compare how the reranker reorders results against the original search. The new score is on `match.score`, the field on `match.document.reranking_field`, and the ordered collection on `reranked_results.data`.

In [22]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

Question: 'patient needs knee surgery'

Reranked Results:
   1. ID: P095
 Score: 0.0131212715
 Reranking Field: symptoms: back pain; treatment: physical therapy

   2. ID: P047
 Score: 0.0131212715
 Reranking Field: symptoms: back pain; treatment: physical therapy

   3. ID: P0100
 Score: 0.0013993983
 Reranking Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain

